# NatGasMoE -- Optuna Optimization & Training

Regime-Aware Heterogeneous Mixture of Experts (TCN + MDN) for natural gas storage forecasting.

**Pipeline:**
1. Load NG futures OHLCV via yfinance
2. Pull EIA weekly storage + fit ConsensusForecast for surprise features
3. Build datasets via `NGMoEDataBuilder` (48 technicals x_seq + 12 regime ae_input)
4. Optuna search over MoE architecture (experts, latent dim, channels, dropout, lr)
5. Train final `NatGasMoE` with best params using `MoELoss`
6. Diagnostics: router utilization, regime clustering, loss breakdown

In [ ]:
import os
import sys
import copy
import time as _time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

warnings.filterwarnings('ignore')

# --- CTAFlow ---
from CTAFlow.models.deep_learning.multi_branch.ng_moe import (
    MoEConfig,
    NatGasMoE,
    MoELoss,
)
from CTAFlow.models.deep_learning.multi_branch.ng_moe_dataset import (
    NGMoEDataConfig,
    NGMoEDataBuilder,
    NGMoEWindowDataset,
    REGIME_COLS,
    build_datasets,
)

# --- macrOS-Int ---
sys.path.insert(0, r'C:\Users\nicho\PycharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import (
    NatGasStorageForecaster,
    fetch_storage_data,
    ConsensusForecast,
    DEFAULT_WEATHER_HDF,
    DEFAULT_EIA_HDF,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
SAVE_DIR = Path(r'C:\Users\nicho\PycharmProjects\CTAFlow\outputs\ng_moe')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EIA_HDF = str(DEFAULT_EIA_HDF)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"EIA HDF  : {EIA_HDF}")
print(f"Save dir : {SAVE_DIR}")

## 1. Load Price Data

In [ ]:
import yfinance as yf

price_df = yf.download("NG=F", start="2010-01-01")
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)
price_df.columns = [c.lower() for c in price_df.columns]
price_df = price_df.dropna(subset=['close'])

print(f'Price data : {price_df.shape}')
print(f'Date range : {price_df.index[0].date()} to {price_df.index[-1].date()}')
price_df.tail(3)

## 2. Load EIA Storage + ConsensusForecast

In [ ]:
START = price_df.index[0].strftime('%Y-%m')
END   = price_df.index[-1].strftime('%Y-%m')

# Try HDF cache first, fall back to API
eia_cache    = NatGasStorageForecaster.load_eia_cache(hdf_path=EIA_HDF)
storage_wkly = eia_cache.get('storage')

if storage_wkly is not None and not storage_wkly.empty:
    print(f'EIA storage loaded from cache : {storage_wkly.shape}')
else:
    ng_helper    = NatGasHelper()
    storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
    NatGasStorageForecaster.save_eia_cache(storage=storage_wkly, hdf_path=EIA_HDF)
    print(f'EIA storage fetched from API : {storage_wkly.shape}')

print(f'Columns : {storage_wkly.columns.tolist()}')
storage_wkly.tail(3)

In [ ]:
# ConsensusForecast for surprise features
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']], how='left'
    )
    print(f'ConsensusForecast added: {surprise_df.columns.tolist()}')
except Exception as e:
    print(f'ConsensusForecast failed ({e}) -- using rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly storage columns: {storage_wkly.columns.tolist()}')
storage_wkly.tail(4)

## 3. Build Datasets

`build_datasets()` runs the full `NGMoEDataBuilder` pipeline:
- 48 technical features (x_seq) -- returns, volatility, momentum, microstructure, temporal, storage daily
- 12 regime features (ae_input) -- storage-driven 5yr band position, forecast vs seasonal, seasonal phase
- Target: next week's storage_change (shifted -1 week)
- Sampled at Monday close only

In [ ]:
data_cfg = NGMoEDataConfig(
    seq_len=20,
    ae_window=21,
    target='storage_change',
)

train_ds, val_ds, test_ds, meta = build_datasets(
    price_df, storage_wkly,
    config=data_cfg,
    train_frac=0.70,
    val_frac=0.15,
    monday_only=True,
)

n_features = meta['n_features']
print(f'Features     : {n_features}')
print(f'Regime cols  : {len(meta["regime_cols"])}')
print(f'Train samples: {len(train_ds)}')
print(f'Val samples  : {len(val_ds)}')
print(f'Test samples : {len(test_ds)}')

for split_name, (start, end, count) in meta['splits'].items():
    print(f'  {split_name:5s}: {start.date()} - {end.date()}  ({count} Mondays)')

# Smoke test one sample
x_seq, ae_input, y_ret, y_std = train_ds[0]
print(f'\nSample shapes: x_seq={tuple(x_seq.shape)}, ae_input={tuple(ae_input.shape)}, '
      f'y_ret={y_ret.item():.4f}, y_std={y_std.item():.4f}')

## 4. Optuna Hyperparameter Search

Search over MoE architecture and training hyperparameters:
- VAE latent dimension, hidden dimension
- Number of TCN / MDN experts, top-k routing
- TCN channel width, MDN hidden dims, MDN components
- Dropout, learning rate, weight decay, loss weights
- Batch size

Each trial trains with early stopping and reports val total loss.
MedianPruner kills underperforming trials early.

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA = True
except ImportError:
    print('optuna not installed -- skipping search, using defaults')
    _OPTUNA = False

N_TRIALS       = 40
MAX_EPOCHS_OPT = 80
OPT_PATIENCE   = 10
OPT_BATCH_SIZES = [16, 32, 64]

# Track results
_trial_log = []


def _make_loaders(train_ds, val_ds, bs):
    tr_dl = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True)
    va_dl = DataLoader(val_ds,   batch_size=bs, shuffle=False, drop_last=False)
    return tr_dl, va_dl


def objective(trial):
    t0 = _time.time()

    # --- Architecture ---
    d_latent       = trial.suggest_categorical('d_latent', [16, 32, 64])
    d_ae_hidden    = trial.suggest_categorical('d_ae_hidden', [64, 128, 256])
    n_tcn_experts  = trial.suggest_int('n_tcn_experts', 2, 5)
    n_mdn_experts  = trial.suggest_int('n_mdn_experts', 1, 4)
    top_k          = trial.suggest_int('top_k', 2, min(n_tcn_experts + n_mdn_experts, 5))
    tcn_width      = trial.suggest_categorical('tcn_width', [32, 64, 128])
    tcn_depth      = trial.suggest_int('tcn_depth', 2, 4)
    mdn_hidden     = trial.suggest_categorical('mdn_hidden', [32, 64, 128])
    mdn_n_comp     = trial.suggest_int('mdn_n_components', 2, 6)
    shared_dim     = trial.suggest_categorical('shared_expert_dim', [32, 64, 128])
    dropout        = trial.suggest_float('dropout', 0.05, 0.4)

    # --- Training ---
    lr             = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    wd             = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    bs             = trial.suggest_categorical('batch_size', OPT_BATCH_SIZES)

    # --- Loss weights ---
    nll_w          = trial.suggest_float('nll_weight', 0.01, 0.5, log=True)
    bal_w          = trial.suggest_float('load_balance_weight', 0.001, 0.1, log=True)
    ent_w          = trial.suggest_float('entropy_reg_weight', 0.001, 0.1, log=True)
    kl_w           = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)

    cfg = MoEConfig(
        n_features=n_features,
        seq_len=data_cfg.seq_len,
        f_ae=12,
        ae_window=data_cfg.ae_window,
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_w,
        n_tcn_experts=n_tcn_experts,
        n_mdn_experts=n_mdn_experts,
        top_k=top_k,
        shared_expert_dim=shared_dim,
        tcn_channels=[tcn_width] * tcn_depth,
        mdn_hidden_dims=[mdn_hidden, mdn_hidden // 2],
        mdn_n_components=mdn_n_comp,
        dropout=dropout,
        load_balance_weight=bal_w,
        entropy_reg_weight=ent_w,
        nll_weight=nll_w,
    )

    model   = NatGasMoE(cfg).to(DEVICE)
    loss_fn = MoELoss(cfg)
    opt     = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS_OPT)

    n_params = sum(p.numel() for p in model.parameters())
    tr_dl, va_dl = _make_loaders(train_ds, val_ds, bs)

    best_val, patience_cnt, final_epoch = float('inf'), 0, 0

    for epoch in range(1, MAX_EPOCHS_OPT + 1):
        final_epoch = epoch

        # Train
        model.train()
        for x_seq, ae_in, y_ret, y_std in tr_dl:
            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            opt.zero_grad()
            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std)
            losses['total_loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_seq, ae_in, y_ret, y_std in va_dl:
                x_seq = x_seq.to(DEVICE)
                ae_in = ae_in.to(DEVICE)
                y_ret = y_ret.to(DEVICE)
                y_std = y_std.to(DEVICE)
                out = model(x_seq, ae_in)
                losses = loss_fn(out, y_ret, y_std)
                val_losses.append(losses['total_loss'].item())

        v = float(np.mean(val_losses))
        trial.report(v, epoch)

        if trial.should_prune():
            elapsed = _time.time() - t0
            _trial_log.append({
                'trial': trial.number, 'val_loss': v, 'best_val': best_val,
                'epochs': final_epoch, 'status': 'PRUNED', 'time': elapsed,
                'n_tcn': n_tcn_experts, 'n_mdn': n_mdn_experts, 'top_k': top_k,
                'd_latent': d_latent, 'tcn': f'{tcn_width}x{tcn_depth}',
                'mdn_nc': mdn_n_comp, 'lr': lr, 'dropout': dropout,
                'bs': bs, 'params': n_params,
            })
            print(f'  Trial {trial.number:3d} | PRUNED  ep {final_epoch:2d} | '
                  f'val={v:.4f} | tcn={n_tcn_experts} mdn={n_mdn_experts} '
                  f'k={top_k} lat={d_latent} | {elapsed:.1f}s')
            raise optuna.TrialPruned()

        if v < best_val:
            best_val, patience_cnt = v, 0
        else:
            patience_cnt += 1
        if patience_cnt >= OPT_PATIENCE:
            break

    elapsed = _time.time() - t0
    is_best = best_val <= (study.best_value if len(study.trials) > 0 else float('inf'))
    tag = ' ** NEW BEST **' if is_best else ''

    _trial_log.append({
        'trial': trial.number, 'val_loss': best_val, 'best_val': best_val,
        'epochs': final_epoch, 'status': 'COMPLETE', 'time': elapsed,
        'n_tcn': n_tcn_experts, 'n_mdn': n_mdn_experts, 'top_k': top_k,
        'd_latent': d_latent, 'tcn': f'{tcn_width}x{tcn_depth}',
        'mdn_nc': mdn_n_comp, 'lr': lr, 'dropout': dropout,
        'bs': bs, 'params': n_params,
    })
    print(f'  Trial {trial.number:3d} | val={best_val:.4f} | ep {final_epoch:2d} | '
          f'tcn={n_tcn_experts} mdn={n_mdn_experts} k={top_k} lat={d_latent} '
          f'ch={tcn_width}x{tcn_depth} mc={mdn_n_comp} '
          f'dr={dropout:.2f} lr={lr:.1e} wd={wd:.1e} bs={bs} '
          f'({n_params:,} params) | {elapsed:.1f}s{tag}')
    return best_val


if _OPTUNA:
    study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=15),
    )
    print(f'Starting Optuna search: {N_TRIALS} trials, {MAX_EPOCHS_OPT} max epochs\n')
    print(f'{"Trial":>7} | {"Val Loss":>9} | {"Ep":>3} | '
          f'{"TCN":>3} {"MDN":>3} {"k":>2} {"lat":>3} {"Channels":>8} {"mc":>2} '
          f'{"dr":>5} {"lr":>8} {"wd":>8} {"bs":>3} {"Params":>8} | {"Time":>5}')
    print('-' * 110)

    study.optimize(objective, n_trials=N_TRIALS)

    # --- Summary ---
    print('\n' + '=' * 110)
    print('SEARCH COMPLETE')
    print('=' * 110)
    completed = [t for t in _trial_log if t['status'] == 'COMPLETE']
    pruned    = [t for t in _trial_log if t['status'] == 'PRUNED']
    print(f'  Completed: {len(completed)}  |  Pruned: {len(pruned)}  |  Total: {len(_trial_log)}')
    total_time = sum(t['time'] for t in _trial_log)
    print(f'  Total search time: {total_time / 60:.1f} min')
    if completed:
        vals = [t['val_loss'] for t in completed]
        print(f'  Val loss range: [{min(vals):.4f}, {max(vals):.4f}]  '
              f'median={np.median(vals):.4f}')

    # Top 5
    print(f'\nTop 5 trials:')
    top5 = sorted(completed, key=lambda t: t['val_loss'])[:5]
    for rank, t in enumerate(top5, 1):
        print(f'  #{rank}  Trial {t["trial"]:3d} | val={t["val_loss"]:.4f} | '
              f'tcn={t["n_tcn"]} mdn={t["n_mdn"]} k={t["top_k"]} lat={t["d_latent"]} '
              f'{t["tcn"]} mc={t["mdn_nc"]} lr={t["lr"]:.1e} '
              f'dr={t["dropout"]:.2f} bs={t["bs"]} ({t["params"]:,} params)')

    best = study.best_params
    print(f'\nBest val loss: {study.best_value:.4f}')
    print(f'Best params  : {best}')

In [ ]:
# Build final config from best Optuna params (or use defaults if no Optuna)
if _OPTUNA:
    bp = study.best_params
    final_cfg = MoEConfig(
        n_features=n_features,
        seq_len=data_cfg.seq_len,
        f_ae=12,
        ae_window=data_cfg.ae_window,
        d_latent=bp['d_latent'],
        d_ae_hidden=bp['d_ae_hidden'],
        kl_weight=bp['kl_weight'],
        n_tcn_experts=bp['n_tcn_experts'],
        n_mdn_experts=bp['n_mdn_experts'],
        top_k=bp['top_k'],
        shared_expert_dim=bp['shared_expert_dim'],
        tcn_channels=[bp['tcn_width']] * bp['tcn_depth'],
        mdn_hidden_dims=[bp['mdn_hidden'], bp['mdn_hidden'] // 2],
        mdn_n_components=bp['mdn_n_components'],
        dropout=bp['dropout'],
        load_balance_weight=bp['load_balance_weight'],
        entropy_reg_weight=bp['entropy_reg_weight'],
        nll_weight=bp['nll_weight'],
    )
    FINAL_LR = bp['lr']
    FINAL_WD = bp['weight_decay']
    FINAL_BS = bp['batch_size']
else:
    final_cfg = MoEConfig(
        n_features=n_features,
        seq_len=data_cfg.seq_len,
        f_ae=12,
        ae_window=data_cfg.ae_window,
    )
    FINAL_LR = 5e-4
    FINAL_WD = 1e-4
    FINAL_BS = 32

print(f'Final config:')
print(f'  TCN experts: {final_cfg.n_tcn_experts}  MDN experts: {final_cfg.n_mdn_experts}  top_k: {final_cfg.top_k}')
print(f'  d_latent: {final_cfg.d_latent}  tcn_channels: {final_cfg.tcn_channels}')
print(f'  mdn_components: {final_cfg.mdn_n_components}  mdn_hidden: {final_cfg.mdn_hidden_dims}')
print(f'  dropout: {final_cfg.dropout:.2f}  lr: {FINAL_LR:.1e}  wd: {FINAL_WD:.1e}  bs: {FINAL_BS}')

## 5. Optuna Visualization

In [ ]:
if _OPTUNA and study is not None:
    trial_df = pd.DataFrame(_trial_log)
    display(trial_df.sort_values('val_loss').head(10))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1. Optimization history
    comp = trial_df[trial_df['status'] == 'COMPLETE'].copy()
    best_so_far = comp['val_loss'].expanding().min()
    axes[0].scatter(comp['trial'], comp['val_loss'], s=20, alpha=0.6, label='trial loss')
    axes[0].plot(comp['trial'].values, best_so_far.values,
                 color='red', lw=1.5, label='best so far')
    axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val Loss')
    axes[0].set_title('Optimization History'); axes[0].legend(fontsize=8)

    # 2. Param importances
    if len(comp) >= 5:
        try:
            imp = optuna.importance.get_param_importances(study)
            names = list(imp.keys())[:10]
            vals  = list(imp.values())[:10]
            axes[1].barh(names, vals, color='steelblue', edgecolor='black')
            axes[1].set_xlabel('Importance')
            axes[1].set_title('Top 10 Hyperparameter Importance')
        except Exception:
            axes[1].text(0.5, 0.5, 'Not enough data\nfor importance',
                         ha='center', va='center', transform=axes[1].transAxes)

    # 3. Time per trial
    axes[2].bar(trial_df['trial'], trial_df['time'],
                color=['steelblue' if s == 'COMPLETE' else 'lightcoral'
                       for s in trial_df['status']],
                edgecolor='black', linewidth=0.3)
    axes[2].set_xlabel('Trial'); axes[2].set_ylabel('Seconds')
    axes[2].set_title('Time per Trial (red = pruned)')

    plt.tight_layout()
    plt.show()

## 6. Final Model Training

Train with best params, full early stopping, and detailed loss tracking.

In [ ]:
MAX_EPOCHS = 200
PATIENCE   = 20

model   = NatGasMoE(final_cfg).to(DEVICE)
loss_fn = MoELoss(final_cfg)
optimizer = optim.AdamW(model.parameters(), lr=FINAL_LR, weight_decay=FINAL_WD)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

train_loader = DataLoader(train_ds, batch_size=FINAL_BS, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=FINAL_BS, shuffle=False, drop_last=False)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params : {total_params:,}')
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')
print(model)

In [ ]:
loss_keys = ['total_loss', 'return_loss', 'vol_loss', 'mdn_nll_loss',
             'balance_loss', 'entropy_loss', 'ae_recon_loss', 'ae_kl_loss']
history = {f'train_{k}': [] for k in loss_keys}
history.update({f'val_{k}': [] for k in loss_keys})
history['shared_gate'] = []

best_val_loss = float('inf')
best_state    = None
patience_cnt  = 0

for epoch in range(1, MAX_EPOCHS + 1):

    # --- Train ---
    model.train()
    epoch_train = {k: [] for k in loss_keys}
    for x_seq, ae_in, y_ret, y_std in train_loader:
        x_seq = x_seq.to(DEVICE)
        ae_in = ae_in.to(DEVICE)
        y_ret = y_ret.to(DEVICE)
        y_std = y_std.to(DEVICE)

        optimizer.zero_grad()
        out = model(x_seq, ae_in)
        losses = loss_fn(out, y_ret, y_std)
        losses['total_loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k in loss_keys:
            epoch_train[k].append(losses[k].item())
    scheduler.step()

    # --- Validate ---
    model.eval()
    epoch_val = {k: [] for k in loss_keys}
    gate_vals = []
    with torch.no_grad():
        for x_seq, ae_in, y_ret, y_std in val_loader:
            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std)
            for k in loss_keys:
                epoch_val[k].append(losses[k].item())
            gate_vals.append(out['shared_gate'].item())

    for k in loss_keys:
        history[f'train_{k}'].append(float(np.mean(epoch_train[k])))
        history[f'val_{k}'].append(float(np.mean(epoch_val[k])))
    history['shared_gate'].append(float(np.mean(gate_vals)))

    v = history['val_total_loss'][-1]
    if v < best_val_loss:
        best_val_loss = v
        best_state    = copy.deepcopy(model.state_dict())
        patience_cnt  = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0:
        lr_now = optimizer.param_groups[0]['lr']
        gate   = history['shared_gate'][-1]
        print(f'Epoch {epoch:3d} | '
              f'T={history["train_total_loss"][-1]:.4f} V={v:.4f} | '
              f'ret={history["val_return_loss"][-1]:.4f} '
              f'vol={history["val_vol_loss"][-1]:.4f} '
              f'nll={history["val_mdn_nll_loss"][-1]:.4f} '
              f'bal={history["val_balance_loss"][-1]:.4f} '
              f'ae={history["val_ae_recon_loss"][-1]:.4f} | '
              f'gate={gate:.3f} lr={lr_now:.2e} pat={patience_cnt}/{PATIENCE}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

## 7. Training Diagnostics -- Loss Breakdown

In [ ]:
ep = range(1, len(history['train_total_loss']) + 1)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Total loss
axes[0, 0].plot(ep, history['train_total_loss'], label='Train')
axes[0, 0].plot(ep, history['val_total_loss'],   label='Val')
axes[0, 0].set_title('Total Loss'); axes[0, 0].legend(); axes[0, 0].set_xlabel('Epoch')

# Return loss (Huber)
axes[0, 1].plot(ep, history['train_return_loss'], label='Train')
axes[0, 1].plot(ep, history['val_return_loss'],   label='Val')
axes[0, 1].set_title('Return Loss (Huber)'); axes[0, 1].legend()

# Vol loss
axes[0, 2].plot(ep, history['train_vol_loss'], label='Train')
axes[0, 2].plot(ep, history['val_vol_loss'],   label='Val')
axes[0, 2].set_title('Vol Loss (MSE log-std)'); axes[0, 2].legend()

# MDN NLL
axes[1, 0].plot(ep, history['val_mdn_nll_loss'], color='purple')
axes[1, 0].set_title('MDN NLL (Val)'); axes[1, 0].set_xlabel('Epoch')

# Balance + Entropy
axes[1, 1].plot(ep, history['val_balance_loss'], label='Balance')
axes[1, 1].plot(ep, history['val_entropy_loss'], label='Entropy')
axes[1, 1].set_title('Router Regularization (Val)'); axes[1, 1].legend()

# Shared gate + AE losses
ax2 = axes[1, 2].twinx()
axes[1, 2].plot(ep, history['shared_gate'], color='steelblue', label='Shared gate')
ax2.plot(ep, history['val_ae_recon_loss'], color='darkorange', label='AE recon')
ax2.plot(ep, history['val_ae_kl_loss'],    color='red', linestyle='--', label='AE KL')
axes[1, 2].set_title('Shared Gate & AE Losses')
axes[1, 2].set_ylabel('Gate alpha', color='steelblue')
ax2.set_ylabel('AE Loss', color='darkorange')
lines1, labels1 = axes[1, 2].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1, 2].legend(lines1 + lines2, labels1 + labels2, fontsize=7)

plt.tight_layout()
plt.show()

## 8. Router Utilization -- Expert Selection Analysis

In [ ]:
model.eval()
all_weights = []
all_z_regime = []

with torch.no_grad():
    for x_seq, ae_in, y_ret, y_std in val_loader:
        x_seq = x_seq.to(DEVICE)
        ae_in = ae_in.to(DEVICE)
        out = model(x_seq, ae_in)
        all_weights.append(out['router_weights'].cpu().numpy())
        all_z_regime.append(out['z_regime'].cpu().numpy())

weights_np = np.concatenate(all_weights)   # (N_val, n_experts)
z_regime_np = np.concatenate(all_z_regime) # (N_val, d_latent)

n_experts = weights_np.shape[1]
n_tcn = final_cfg.n_tcn_experts
n_mdn = final_cfg.n_mdn_experts

avg_w = weights_np.mean(axis=0)
expert_labels = [f'TCN_{i}' for i in range(n_tcn)] + [f'MDN_{i}' for i in range(n_mdn)]

print(f'{"Expert":<8} {"Avg Weight":>10} {"Active %":>9} {"Max Weight":>10}')
print('-' * 42)
for i, label in enumerate(expert_labels):
    active_pct = (weights_np[:, i] > 0).mean() * 100
    max_w = weights_np[:, i].max()
    flag = '  <- underused' if avg_w[i] < 0.5 / n_experts else ''
    print(f'{label:<8} {avg_w[i]:>10.4f} {active_pct:>8.1f}% {max_w:>10.4f}{flag}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Average weights
colors = ['steelblue'] * n_tcn + ['darkorange'] * n_mdn
axes[0].bar(expert_labels, avg_w, color=colors, edgecolor='black')
axes[0].axhline(1.0 / n_experts, color='red', linestyle='--', label='uniform')
axes[0].set_ylabel('Average Router Weight')
axes[0].set_title('Expert Utilization (Val)')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=45)

# Weight distribution per expert (box plot)
axes[1].boxplot([weights_np[:, i] for i in range(n_experts)],
                labels=expert_labels, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_ylabel('Router Weight')
axes[1].set_title('Weight Distribution per Expert')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9. Regime Clustering -- VAE Latent Space

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# PCA on z_regime for visualization
pca = PCA(n_components=2)
z_2d = pca.fit_transform(z_regime_np)
print(f'PCA explained variance: {pca.explained_variance_ratio_[0]:.1%}, {pca.explained_variance_ratio_[1]:.1%}')

# K-means clustering on latent space
n_clusters = 4
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=SEED)
labels = km.fit_predict(z_regime_np)

# Get val dates and dominant expert per sample
val_dates = [val_ds.get_date(i) for i in range(len(val_ds))]
dominant_expert = np.argmax(weights_np, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Colored by K-means cluster
scatter = axes[0].scatter(z_2d[:, 0], z_2d[:, 1], c=labels, cmap='tab10',
                          s=15, alpha=0.6)
axes[0].set_title(f'z_regime PCA -- K-Means ({n_clusters} clusters)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# 2. Colored by dominant expert
scatter2 = axes[1].scatter(z_2d[:, 0], z_2d[:, 1], c=dominant_expert,
                           cmap='Set1', s=15, alpha=0.6)
axes[1].set_title('z_regime PCA -- Dominant Expert')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
cbar2 = plt.colorbar(scatter2, ax=axes[1], label='Expert idx')
cbar2.set_ticks(range(n_experts))
cbar2.set_ticklabels(expert_labels)

# 3. Colored by injection/withdrawal season
val_months = np.array([d.month for d in val_dates])
is_injection = ((val_months >= 4) & (val_months <= 10)).astype(float)
scatter3 = axes[2].scatter(z_2d[:, 0], z_2d[:, 1], c=is_injection,
                           cmap='coolwarm', s=15, alpha=0.6)
axes[2].set_title('z_regime PCA -- Season')
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')
plt.colorbar(scatter3, ax=axes[2], label='Injection=1')

plt.tight_layout()
plt.show()

# Cluster stats
print(f'\nCluster   Count   Mean|target|  Mean rv_5d   Season mix (inj%)')
print('-' * 65)
# Get val targets
val_targets = np.array([val_ds[i][2].item() for i in range(len(val_ds))])
for c in range(n_clusters):
    mask = labels == c
    ct = mask.sum()
    mean_abs_tgt = np.abs(val_targets[mask]).mean()
    inj_pct = is_injection[mask].mean() * 100
    print(f'  {c:3d}     {ct:5d}      {mean_abs_tgt:8.2f}                  {inj_pct:5.1f}%')

## 10. Prediction Analysis -- Val + Test

In [ ]:
def evaluate_split(ds, name):
    """Run model on a dataset split, return predictions + actuals."""
    loader = DataLoader(ds, batch_size=64, shuffle=False, drop_last=False)
    preds_ret, preds_std, actuals_ret, actuals_std = [], [], [], []
    dates = []

    model.eval()
    with torch.no_grad():
        for i, (x_seq, ae_in, y_ret, y_std) in enumerate(loader):
            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            out = model(x_seq, ae_in)

            preds_ret.append(out['pred_return'].cpu().numpy())
            preds_std.append(out['pred_std'].cpu().numpy())
            actuals_ret.append(y_ret.numpy())
            actuals_std.append(y_std.numpy())

    preds_ret = np.concatenate(preds_ret)
    preds_std = np.concatenate(preds_std)
    actuals_ret = np.concatenate(actuals_ret)
    actuals_std = np.concatenate(actuals_std)
    dates = [ds.get_date(i) for i in range(len(ds))]

    # Metrics
    mae = np.abs(preds_ret - actuals_ret).mean()
    rmse = np.sqrt(((preds_ret - actuals_ret) ** 2).mean())
    corr = np.corrcoef(preds_ret, actuals_ret)[0, 1]
    dir_mask = np.abs(actuals_ret) > 1e-6
    dir_acc = (np.sign(preds_ret[dir_mask]) == np.sign(actuals_ret[dir_mask])).mean()

    print(f'\n{name} set ({len(ds)} samples):')
    print(f'  MAE  : {mae:.4f}')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  Corr : {corr:.4f}')
    print(f'  Dir%  : {dir_acc:.1%}')

    return pd.DataFrame({
        'date': dates,
        'actual': actuals_ret,
        'pred': preds_ret,
        'pred_std': preds_std,
        'actual_std': actuals_std,
    }).set_index('date')

val_results  = evaluate_split(val_ds, 'Validation')
test_results = evaluate_split(test_ds, 'Test')

In [ ]:
# Predicted vs actual scatter + time series
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax_row, (results, name) in zip(axes, [(val_results, 'Validation'), (test_results, 'Test')]):
    # Scatter
    ax_row[0].scatter(results['actual'], results['pred'], s=10, alpha=0.5)
    lims = [min(results['actual'].min(), results['pred'].min()),
            max(results['actual'].max(), results['pred'].max())]
    ax_row[0].plot(lims, lims, 'r--', lw=1)
    ax_row[0].set_xlabel('Actual'); ax_row[0].set_ylabel('Predicted')
    ax_row[0].set_title(f'{name}: Predicted vs Actual Storage Change')

    # Time series with +/- 1 std band
    ax_row[1].plot(results.index, results['actual'], color='black', lw=0.8,
                   alpha=0.7, label='Actual')
    ax_row[1].plot(results.index, results['pred'], color='steelblue', lw=0.8,
                   alpha=0.8, label='Predicted')
    ax_row[1].fill_between(
        results.index,
        results['pred'] - results['pred_std'],
        results['pred'] + results['pred_std'],
        alpha=0.15, color='steelblue', label='+/- 1 std',
    )
    ax_row[1].set_title(f'{name}: Storage Change Forecast')
    ax_row[1].set_ylabel('Storage Change (Bcf)')
    ax_row[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 11. Save Weights & Config

In [ ]:
import json
import dataclasses

# Save model weights
weights_path = SAVE_DIR / 'ng_moe_best.pt'
torch.save(model.state_dict(), weights_path)
print(f'Weights saved: {weights_path}')

# Save config as JSON
cfg_dict = dataclasses.asdict(final_cfg)
cfg_dict['_training'] = {
    'lr': FINAL_LR,
    'weight_decay': FINAL_WD,
    'batch_size': FINAL_BS,
    'best_val_loss': best_val_loss,
    'n_features': n_features,
    'feature_cols': meta['feature_cols'],
}
cfg_path = SAVE_DIR / 'ng_moe_config.json'
with open(cfg_path, 'w') as f:
    json.dump(cfg_dict, f, indent=2)
print(f'Config saved : {cfg_path}')

# Save Optuna study if available
if _OPTUNA:
    trial_path = SAVE_DIR / 'optuna_trials.csv'
    pd.DataFrame(_trial_log).to_csv(trial_path, index=False)
    print(f'Trials saved : {trial_path}')

print('\nDone.')